# Curating, inspecting, and submitting simulations

In [9]:
import json
import os
import pandas as pd
import numpy as np
import subprocess
import shutil

while os.path.split(os.getcwd())[-1] != 'calvados-slim':
    os.chdir('..')

from src import curation, utils

## Curating simulations

In [10]:
# --- Inputs ---
with open('data/curation/systems.json') as file:
    systems = json.load(file)
with open('data/configurations/sampling.json') as file:
    sampling = json.load(file)

# --- Configuration ---
SYSTEMS_DIR = os.path.abspath('data/curation/systems')
SIM_DIR     = os.path.abspath('sim')


### Speed assessment on CPU
*Running short simulations on CPU to determine how many replicates would be necessary for a fair walltime*

In [55]:
# Setting modes to create runs for
conditions_mode = 'Default'
sampling_mode = 'Speed'
replicates = 1

# Defining eps_go range
eps_gos = [5]

# Running on DeiC
platform = 'CPU'

In [56]:
# Looping over systems
for system in systems:
    system_dir = os.path.join(SYSTEMS_DIR, system['ID'])
    if not os.path.exists(system_dir):
        raise ValueError(f"System files must be precurated in {SYSTEMS_DIR}!")

    # Looping over eps_go scan
    for eps_go in eps_gos:
        eps_dir = os.path.join(SIM_DIR, system['ID'], f"eps_go={eps_go:.2f}")

        # Creating replicate directories
        for rep in range(1, replicates+1):
            rep_path = os.path.join(eps_dir, sampling_mode.lower(), f"rep{rep}")
            if os.path.exists(rep_path):
                continue

            print(f"Creating '{rep_path}'")
            try:
                curation.Simulation.build(
                    system,
                    system_dir,
                    eps_go,
                    rep_path,
                    platform,
                    stride,
                    frames,
                )
            except Exception as e:
                print(f"Failed to create {rep_path}: {e}")
                shutil.rmtree(rep_path)

### Scan simulations
*Running long simulations for many a common set of eps_go values*

In [12]:
# Loading simulation plan
sim_plan = pd.read_excel('data/planning/scan_plan.xlsx', skiprows=4, index_col=0)
sim_plan.sort_values('Simulation speed on CPU [ns/day]', ascending=False, inplace=True)
server = input("Which server is this running on (ROBUST, DeiC, Local)?")
sim_plan = sim_plan[sim_plan['Planned server'] == server]
sim_plan


,Contacts,Simulation speed on CPU [ns/day],Necessary replicates,Planned replicates,Planned server,Note
System,,,,,,
atg8-atg32,27,999.500000,2,1,ROBUST,NaN
stip1-hspa1l,30,994.500000,2,1,ROBUST,NaN
mdm2-p53,45,993.000000,2,1,ROBUST,NaN
rims1-erc1,34,992.000000,2,1,ROBUST,NaN
pard3-cdh5,27,989.000000,2,1,ROBUST,NaN
...,...,...,...,...,...,...
pp1c-ppp1r2,132,124.666667,9,1,ROBUST,NaN
f11-lama1,29,96.500000,11,1,ROBUST,NaN
f11-kng1,31,95.000000,11,1,ROBUST,NaN


In [ ]:
# Setting modes to create runs for
conditions_mode = 'Default'
sampling_mode = 'Refinement'
stride = sampling[sampling_mode]['Stride']

# Defining eps_go range
eps_go_min  = 0
eps_go_max  = 5
eps_go_step = 1
eps_gos = np.arange(eps_go_min, eps_go_max+eps_go_step, eps_go_step)

# Running according to plan
platform = 'CPU' if server == 'DeiC' else 'GPU'

In [59]:
# Looping over systems
for system in systems:
    system_dir = os.path.join(SYSTEMS_DIR, system['ID'])
    if not os.path.exists(system_dir):
        raise ValueError(f"System files must be precurated in {SYSTEMS_DIR}!")

    # Skipping systems that are not included in simulation plan
    if system['ID'] not in sim_plan.index:
        continue

    # Looping over eps_go scan
    for eps_go in eps_gos:
        eps_dir = os.path.join(SIM_DIR, system['ID'], f"eps_go={eps_go:.2f}")

        # System-server specific replicats
        replicates = sim_plan.loc[system['ID'], 'Planned replicates']
        frames = sampling[sampling_mode]['Frames'] // replicates

        # Creating replicate directories
        for rep in range(1, replicates+1):
            rep_path = os.path.join(eps_dir, sampling_mode.lower(), f"rep{rep}")
            if os.path.exists(rep_path):
                continue

            print(f"Creating '{rep_path}'")
            try:
                curation.Simulation.build(
                    system,
                    system_dir,
                    eps_go,
                    rep_path,
                    platform,
                    stride,
                    frames,
                )
            except Exception as e:
                print(f"Failed to create {rep_path}: {e}")
                shutil.rmtree(rep_path)

### Refinement simulations
*Running additional simulations for system-specific eps_go-values of interest*

In [1]:
# Loading simulation plan
refinement_plan = pd.read_csv('data/planning/refinement_priority.csv')
refinement_plan.sort_values('N_eff_minor', ascending=True, inplace=True)
server = input("Which server is this running on (ROBUST, DeiC, Local)?")
refinement_plan


NameError: name 'pd' is not defined

In [ ]:
# Setting modes to create runs for
conditions_mode = 'Default'
sampling_mode = 'Scan'
stride = sampling[sampling_mode]['Stride']

# Running according to plan
platform = 'CPU' if server == 'DeiC' else 'GPU'

In [ ]:
for _, row in refinement_plan.iterrows():

    assert server == 'ROBUST', "Only set up for 1 replicate!"
    frames = sampling[sampling_mode]['Frames'] // replicates
    replicates = 1

    system_id = row['System']
    eps_go = row['eps_go']
    system_dir = os.path.join(SYSTEMS_DIR, system_id)
    
    for system in systems:
        if system['ID'] == system_id:
            break
    else:
        raise ValueError("System not found in curation!")

    # Creating replicate directories
    for rep in range(1, replicates+1):
        rep_path = os.path.join(system_id, eps_go, sampling_mode.lower(), f"rep{rep}")
        if os.path.exists(rep_path):
                continue

        print(f"Creating '{rep_path}'")
        try:
            curation.Simulation.build(
                system,
                system_dir,
                eps_go,
                rep_path,
                platform,
                stride,
                frames,
            )
        except Exception as e:
            print(f"Failed to create {rep_path}: {e}")
            shutil.rmtree(rep_path)

## Inspecting simulations

In [78]:
sims = utils.get_sim_dirs('sim')
sims['Speed'] = (sims.Path + '/' + sims.System + '.log').map(utils.get_log_speed)
sims['Frames'] = (sims.Path + '/' + sims.System + '.dcd').map(utils.get_dcd_frame_count)
sims

,System,eps_go,Sampling mode,Replicate,Path,Speed,Frames
0,afadin-bcr,0.0,Scan,1,sim/afadin-bcr/eps_go=0.00/scan/rep1,1560.480962,10000
1,afadin-bcr,1.0,Scan,1,sim/afadin-bcr/eps_go=1.00/scan/rep1,1440.040080,10000
2,afadin-bcr,2.0,Scan,1,sim/afadin-bcr/eps_go=2.00/scan/rep1,1440.040080,10000
3,afadin-bcr,3.0,Scan,1,sim/afadin-bcr/eps_go=3.00/scan/rep1,1439.378758,10000
4,afadin-bcr,4.0,Scan,1,sim/afadin-bcr/eps_go=4.00/scan/rep1,1439.699399,10000
...,...,...,...,...,...,...,...
2184,ywhab-mlxipl,2.0,Scan,1,sim/ywhab-mlxipl/eps_go=2.00/scan/rep1,16988.577154,10000
2185,ywhab-mlxipl,3.0,Scan,1,sim/ywhab-mlxipl/eps_go=3.00/scan/rep1,16587.575150,10000
2186,ywhab-mlxipl,4.0,Scan,1,sim/ywhab-mlxipl/eps_go=4.00/scan/rep1,16601.202405,10000
2187,ywhab-mlxipl,5.0,Scan,1,sim/ywhab-mlxipl/eps_go=5.00/scan/rep1,16500.000000,10000


### Evaluating need for replicas

In [63]:
# Estimating need for replicates
system_cpu_simspeed = sims[sims['Sampling mode'] == 'Speed'].groupby('System')['Speed'].mean().rename('CPU Simulation speed [ns/day]')
system_cpu_simspeed.to_csv('data/planning/system_cpu_simspeed.csv')
goal_sim_time = 5000 # ns
goal_wall_time = 5 # Days   # Actually 7, but left safety margin
total_wall_time = (goal_sim_time / system_cpu_simspeed)
necessary_replicates = (total_wall_time / goal_wall_time).map(np.ceil).rename("CPU Replicates")
necessary_replicates.value_counts().sort_index()

CPU Replicates
1.0     70
2.0     91
3.0     43
4.0     20
5.0     30
6.0     13
7.0      3
8.0      1
9.0      1
11.0     2
12.0     1
21.0     1
Name: count, dtype: int64

## Submitting simulations

In [79]:
# Criteria options (Sim DF mapping functions)
waiting = lambda row: np.isnan(row['Frames'])
done = lambda row: row['Frames'] in [smode['Frames'] for smode in sampling.values()]
running = lambda row: (not done(row)) & (not waiting(row))
whole_eps_go = lambda row: row['eps_go'] % 1 == 0
specific_eps_gos = lambda ks: lambda row: row['eps_go'] in [round(f, 2) for f in ks]
specific_systems = lambda sys: lambda row: row['System'] in sys
specific_mode = lambda mode: lambda row: row['Sampling mode'] == mode

In [80]:
# Selecting simulations for submission
criteria = [
    waiting
] 

sims_sel = sims.copy()
for criterion in criteria:
    sims_sel = sims_sel[sims_sel.apply(criterion, axis=1)]
sims_sel.sort_values('Frames')

,System,eps_go,Sampling mode,Replicate,Path,Speed,Frames


In [81]:
# Submitting simulations
slurm_script = 'deic_cpu.sh' if server == 'DeiC' else 'robust_gpu.sh'
for _, sim in sims_sel.iterrows():
    break # COMMENT AWAY MANUALLY - SUBMIT WITH CARE
    subprocess.run(
        ['sbatch', slurm_script],
        cwd=sim['Path']
    )